# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## 1. Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
image = a9a80d979c9a40599e168f1291843e84
Get existing dask cluster: 'a9a80d979c9a40599e168f1291843e84'
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/a9a80d979c9a40599e168f1291843e84/status
Dask workers for 'dask-eopf' are up: 2/2


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
# Other imports
import os
import os.path as osp

from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "owner_id": OWNER_ID,
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "owner_id": OWNER_ID,
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "owner_id": OWNER_ID,
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

14:55:29.303 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

14:55:29.305 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.short.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

14:55:29.306 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_configuration.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

14:55:29.307 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

14:55:29.308 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration.yaml'.

14:55:29.310 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_dordop_payload.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

14:55:29.342 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'

In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if cluster_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster.name

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [5]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [6]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

14:55:44.100 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'first-l0-processor/sprint21-first-l0-processor' successfully     │
│ created with id 'd6b09cc8-7071-4def-93b2-ded985333166'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/d6b09cc8-7071-4def-93b2-ded985333166


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor/sprint21-first-l0-processor'



In [7]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor/sprint21-first-l0-processor'


## 3. Run Prefect flow for S1 short data (~1 minute)

In [8]:
output_data_dir = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1_short)

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short'


In [9]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'thundering-wallaby'.
└── UUID: 3d5f560d-76e7-4ff1-913d-dec193b0c8cc
└── Parameters: {'owner_id': 'jgaucher', 'input_config_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/config', 'payload_file': 's1/iw_joborder.short.yaml', 'output_data_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-05-22 14:55:46 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/3d5f560d-76e7-4ff1-913d-dec193b0c8cc
Watching flow run 'thundering-wallaby'...


14:55:51.370 | INFO    | prefect - Flow run is in state 'Pending'
14:56:06.389 | INFO    | prefect - Flow run is in state 'Running'
14:57:13.734 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [10]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

Output products generated on: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short'
Download reports locally: './l0/reports/s1.short'


## 4. Run Prefect flow for S1 full data (~30 minutes)

In [ ]:
output_data_dir = s1["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")
    
    local_report_dir = osp.join("./l0", "reports", "s1")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 5. Run Prefect flow for S3 full data (~20 minutes)

In [ ]:
output_data_dir = s3["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s3)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")

    local_report_dir = osp.join("./l0", "reports", "s3")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [5]:
from importlib import reload
debug_flow = False

In [10]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway, None)
    init_dask_cluster_eopf(scale=4)
    from resources.dask_utils import *
    dask_gateway = dask_gateway_eopf
    dask_client = dask_client_eopf
    dask_cluster = dask_cluster_eopf
    os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster.name

Shutting down cluster '28325791e2ed408cb77fea5f1d033f37' ...
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/a9a80d979c9a40599e168f1291843e84/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/4
Dask workers for 'dask-eopf' are up: 4/4


In [11]:
if debug_flow:
    import first_l0_processor
    reload(first_l0_processor)
    results = await first_l0_processor.first_l0_processor(**s1_short)
    display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


14:53:36.828 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/a22be876-9bdd-4d1d-8356-fa77bcfa69b0

14:53:36.862 | INFO    | Flow run 'heavenly-cuckoo' - Beginning flow run 'heavenly-cuckoo' for flow 'first-l0-processor'

14:53:36.863 | INFO    | Flow run 'heavenly-cuckoo' - View at http://prefect-server:4200/runs/flow-run/a22be876-9bdd-4d1d-8356-fa77bcfa69b0

14:53:36.894 | WARNING | opentelemetry.trace - Overriding of current TracerProvider is not allowed

...

14:54:51.480 | INFO    | Task run 'dummy_catalog_save-1b0' - Start catalog saving

14:54:52.586 | INFO    | Task run 'dummy_catalog_save-1b0' - End (dummy) catalog saving:

14:54:52.591 | INFO    | Task run 'dummy_catalog_save-1b0' - Finished in state Completed()

14:54:52.626 | INFO    | Flow run 'heavenly-cuckoo' - Finished in state Completed()

{}